코드 training-loop

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# 1) 모델 정의
class MLP(nn.Module):
    def __init__(self, d_in, d_hidden, d_out):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)
    def forward(self, x):
        h = torch.relu(self.fc1(x))
        return self.fc2(h)

model = MLP(d_in=784, d_hidden=128, d_out=10).to(device)

# 2) 손실 함수와 옵티마이저
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3) 학습 루프
for epoch in range(num_epochs):
    model.train()
    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        # (a) 기울기 초기화
        optimizer.zero_grad()
        # (b) 순전파
        logits = model(x_batch)
        loss = criterion(logits, y_batch)
        # (c) 역전파 (자동 미분)
        loss.backward()
        # (d) 매개변수 갱신
        optimizer.step()
    # 검증 (생략)

코드 image-preprocessing

In [ ]:
import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# 학습용 (증강 포함)
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

# 검증·시험용 (증강 없음)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

# 데이터셋과 로더
train_dataset = datasets.ImageFolder(root='data/train',
                                       transform=train_transform)
val_dataset = datasets.ImageFolder(root='data/val',
                                     transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=64,
                            shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64,
                          shuffle=False, num_workers=4)
print(f"학습: {len(train_dataset)}, 검증: {len(val_dataset)}")
print(f"클래스: {train_dataset.classes}")

코드 transfer-learning

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_CLASSES = 4   # 광고·뉴스·드라마·예능

# 1) 사전학습 ResNet-50 로드
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# 2) 마지막 완전연결 층 교체
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, NUM_CLASSES)
model = model.to(device)

# 3) 손실 함수와 옵티마이저
criterion = nn.CrossEntropyLoss()
# 백본과 분류 층에 다른 학습률 적용
optimizer = optim.AdamW([
    {'params': model.fc.parameters(), 'lr': 1e-3},
    {'params': [p for n, p in model.named_parameters()
                if 'fc' not in n], 'lr': 1e-4}
], weight_decay=1e-4)

scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

# 4) 학습 루프
best_val_acc = 0.0
for epoch in range(20):
    # 학습
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    scheduler.step()

    # 검증
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    val_acc = correct / total
    print(f"Epoch {epoch+1}: 검증 정확도 = {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pt')

코드 gradcam

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import matplotlib.pyplot as plt

# 마지막 합성곱 층을 대상으로 Grad-CAM 적용
target_layer = model.layer4[-1]
cam = GradCAM(model=model, target_layers=[target_layer])

# 한 이미지에 대해 모든 클래스의 주목 영역 시각화
for img_tensor, true_label in val_loader:
    img = img_tensor[0:1].to(device)
    pred_class = model(img).argmax(dim=1).item()
    targets = [ClassifierOutputTarget(pred_class)]
    grayscale_cam = cam(input_tensor=img, targets=targets)[0]

    # 원본 이미지 위에 히트맵 오버레이
    plt.imshow(img[0].cpu().permute(1,2,0).numpy())
    plt.imshow(grayscale_cam, cmap='jet', alpha=0.5)
    plt.title(f"예측: {classes[pred_class]}")
    plt.axis('off'); plt.show()
    break